# Chinchilla ratio table

For each candidate total token budget (Gutenberg + FineWeb-Edu filler) and each model size, shows what multiple of the Chinchilla-optimal budget (20 tokens/param) that total represents.

In [111]:
import pandas as pd

def ratio(budget_b, model_b):
    d_opt = CHINCHILLA_TOKENS_PER_PARAM * model_b
    return budget_b / d_opt

def make_ratio_table(budgets_b, model_sizes_b, show_cells=None, hide_cells=None):
    budgets_b = sorted(budgets_b)
    index = [b / 1e9 for b in budgets_b]
    table = pd.DataFrame(
        {name: [ratio(b, n) for b in budgets_b] for name, n in model_sizes_b.items()},
        index=index,
    )
    table.index.name = "budget (B tokens)"

    if show_cells is not None:
        mask = pd.DataFrame(False, index=table.index, columns=table.columns)
        for idx, cols in show_cells.items():
            mask.loc[idx, cols] = True
        table = table.where(mask)

    if hide_cells is not None:
        mask = pd.DataFrame(True, index=table.index, columns=table.columns)
        for idx, cols in hide_cells.items():
            mask.loc[idx, cols] = False
        table = table.where(mask)

    return table.style.format("{:.2f}x", na_rep="").format_index("{:.4f}")

def find_ratio_coincidences(budgets_b, model_sizes_b, rel_tol=0.01, min_group_size=2):
    entries = sorted(
        (ratio(b, n), b / 1e9, name)
        for b in budgets_b
        for name, n in model_sizes_b.items()
    )

    groups = [[entries[0]]]
    for entry in entries[1:]:
        prev_ratio = groups[-1][-1][0]
        if abs(entry[0] - prev_ratio) / prev_ratio <= rel_tol:
            groups[-1].append(entry)
        else:
            groups.append([entry])

    return [g for g in groups if len(g) >= min_group_size]

def show_cells_from_coincidences(budgets_b, model_sizes_b, rel_tol=0.01, min_group_size=2):
    show_cells = {}
    groups = find_ratio_coincidences(budgets_b, model_sizes_b, rel_tol, min_group_size)
    for group in groups:
        for _, budget_b, name in group:
            show_cells.setdefault(budget_b, []).append(name)
    return show_cells

def generate_magic_budgets(seed_budgets_b, model_sizes_b, min_budget_b, max_budget_b, max_hops=2, dedup_rel_tol=1e-4):
    # every "magic" budget is seed * (Ni/Nj) for some pair of model sizes: that's the only way
    # two different (budget, model) cells can land on the exact same chinchilla ratio.
    # max_hops caps how many such multiplications can be chained, since chaining them freely
    # generates a dense, unbounded set of near-duplicate values instead of a useful shortlist.
    sizes = list(model_sizes_b.values())
    multipliers = {sizes[i] / sizes[j] for i in range(len(sizes)) for j in range(len(sizes)) if i != j}

    found = list(seed_budgets_b)
    frontier = list(seed_budgets_b)
    for _ in range(max_hops):
        new_frontier = []
        for b in frontier:
            for m in multipliers:
                candidate = b * m
                if min_budget_b <= candidate <= max_budget_b:
                    if not any(abs(candidate - f) / f < dedup_rel_tol for f in found):
                        found.append(candidate)
                        new_frontier.append(candidate)
        frontier = new_frontier

    return sorted(found)

def complete_coincidence_groups(budgets_b, model_sizes_b, min_budget_b, max_budget_b, rel_tol=0.01, min_group_size=2, dedup_rel_tol=1e-4):
    # if N-1 models already coincide on a ratio, adds the missing model's budget for
    # that same ratio too (if it fits in range), instead of leaving it out.
    groups = find_ratio_coincidences(budgets_b, model_sizes_b, rel_tol, min_group_size)
    new_budgets_b = list(budgets_b)
    show_cells = {}

    def add_show(b, name):
        cell = show_cells.setdefault(b / 1e9, [])
        if name not in cell:
            cell.append(name)

    for group in groups:
        for _, budget_b_billions, name in group:
            add_show(budget_b_billions * 1e9, name)

        mean_ratio = sum(r for r, _, _ in group) / len(group)
        present_models = {name for _, _, name in group}
        for name, n in model_sizes_b.items():
            if name in present_models:
                continue
            candidate = mean_ratio * CHINCHILLA_TOKENS_PER_PARAM * n
            existing_match = next((b for b in new_budgets_b if abs(candidate - b) / b < dedup_rel_tol), None)
            if existing_match is not None:
                add_show(existing_match, name)
            elif min_budget_b <= candidate <= max_budget_b:
                new_budgets_b.append(candidate)
                add_show(candidate, name)

    return sorted(new_budgets_b), show_cells

In [103]:
CHINCHILLA_TOKENS_PER_PARAM = 20

# total token budgets, in billions (gutenberg + fineweb filler)
GUTENBERG_BUDGET = 2762833920
FINEWEBEDU_BUDGETS = [0, 2, 5, 10, 20, 30, 40, 60, 80, 120, 160]
BUDGETS_B = [b*1000000000 + GUTENBERG_BUDGET for b in FINEWEBEDU_BUDGETS]
OTHER_BUDGETS  = [28.677, 83, 64.250, 80.313, 144.564]
OTHER_BUDGETS  = []
BUDGETS_B += [b*1000000000 for b in OTHER_BUDGETS]

# model sizes, in billions of params
MODEL_SIZES_B = {
    "1.236B": 1236076544,
    "3.563B (TBD)": 3563498496,
    "8.031B": 8031309824,
}

make_ratio_table(BUDGETS_B, MODEL_SIZES_B)

,1.236B,3.563B (TBD),8.031B
budget (B tokens),,,
2.7628,0.11x,0.04x,0.02x
4.7628,0.19x,0.07x,0.03x
7.7628,0.31x,0.11x,0.05x
12.7628,0.52x,0.18x,0.08x
22.7628,0.92x,0.32x,0.14x
32.7628,1.33x,0.46x,0.20x
42.7628,1.73x,0.60x,0.27x
62.7628,2.54x,0.88x,0.39x
82.7628,3.35x,1.16x,0.52x


In [130]:
selected_budgets = [3.461, 7.664, 9.889, 14.833, 22.249, 24.722, 28.677, 42.7628, 49.443, 64.2500, 71.270, 96.376, 123.297, 144.564, 160.626]

# only show these (budget, model) cells; everything else is left blank
# show_cells = {
#     28.677: ["1.236B"],
#     64.2500: ["3.563B (TBD)"],
#     160.626: ["8.031B"],
# }
hide_cells = {
    3.461: ["3.563B (TBD)", "8.031B"],
    7.6640: ["3.563B (TBD)", "8.031B"],
    9.8890: ["8.031B"],
    14.833: ["3.563B (TBD)", "8.031B"],
    24.722: ["3.563B (TBD)", "8.031B"],
    28.6770: ["1.236B", "8.031B"],
    42.7628: ["8.031B"],
    49.4430: ["3.563B (TBD)"],
    71.270: ["1.236B", "8.031B"],
    123.297: ["8.031B"],
    144.5640: ["1.236B"]
}

make_ratio_table([b * 10e8 for b in selected_budgets], MODEL_SIZES_B, hide_cells=hide_cells)

,1.236B,3.563B (TBD),8.031B
budget (B tokens),,,
3.4610,0.14x,,
7.6640,0.31x,,
9.8890,0.40x,0.14x,
14.8330,0.60x,,
22.2490,0.90x,0.31x,0.14x
24.7220,1.00x,,
28.6770,,0.40x,
42.7628,1.73x,0.60x,
49.4430,2.00x,,0.31x


In [100]:
selected_budgets_2 = [12.763, 28.677, 42.763, 64.250, 82.763, 122.763, 160.626]
selected_budgets_2 = [14.8330, 24.7220, 42.7628, 49.443, 71.2700, 96.3760, 123.297, 142.540, 160.6260]
# [1.73, 0.6, 0.27]
budgets_b_2 = [b * 10e8 for b in selected_budgets_2]
#
# show_cells = {
#     12.763: ["3.563B (TBD)", "1.236B"],
#     28.677: ["8.031B", "3.563B (TBD)", "1.236B"],
#     42.763: ["1.236B"],
#     64.250: ["8.031B", "3.563B (TBD)"],
#     82.763: ["8.031B", "3.563B (TBD)", "1.236B"],
#     122.763: ["3.563B (TBD)"],
#     160.626: ["8.031B"],
# }

hide_cells = {
    49.4430: ["3.563B (TBD)", "8.031B"],
    71.2700: ["1.236B", "8.031B"],
    96.3760: ["1.236B", "3.563B (TBD)"],
    142.5400: ["1.236B"],
    160.6260: ["3.563B (TBD)"]
}

make_ratio_table(budgets_b_2, MODEL_SIZES_B, hide_cells=hide_cells)

,1.236B,3.563B (TBD),8.031B
budget (B tokens),,,
14.8330,0.60x,0.21x,0.09x
24.7220,1.00x,0.35x,0.15x
42.7628,1.73x,0.60x,0.27x
49.4430,2.00x,,
71.2700,,1.00x,
96.3760,,,0.60x
123.2970,4.99x,1.73x,0.77x
142.5400,,2.00x,0.89x
160.6260,6.50x,,1.00x


## Reversed: budget needed per ratio

For each candidate Chinchilla ratio and each model size, shows the total token budget (Gutenberg + FineWeb-Edu filler) needed to hit that ratio.

In [115]:
RATIOS = [0.14, 0.31, 0.4, 0.6, 0.9, 1, 1.16, 1.73, 1.75, 2, 2.25, 3.5]

def budget_for_ratio(r, model_b):
    return r * CHINCHILLA_TOKENS_PER_PARAM * model_b

budget_table = pd.DataFrame(
    {name: [budget_for_ratio(r, n) / 1e9 for r in RATIOS] for name, n in MODEL_SIZES_B.items()},
    index=RATIOS,
)
budget_table.index.name = "ratio"
budget_table.style.format("{:.3f}B").format_index("{:.2f}x")

,1.236B,3.563B (TBD),8.031B
ratio,,,
0.14x,3.461B,9.978B,22.488B
0.31x,7.664B,22.094B,49.794B
0.40x,9.889B,28.508B,64.250B
0.60x,14.833B,42.762B,96.376B
0.90x,22.249B,64.143B,144.564B
1.00x,24.722B,71.270B,160.626B
1.16x,28.677B,82.673B,186.326B
1.73x,42.768B,123.297B,277.883B
1.75x,43.263B,124.722B,281.096B


## Ratio coincidences

Different (budget, model) pairs can land on the same ratio, because for a fixed budget the three models' ratios are always in the fixed proportion `N2/N1`, `N3/N2`, `N3/N1` (~2.883, ~2.254, ~6.497 for the current sizes). Whenever two budgets happen to be related by one of those constants, a "clean" ratio picked for one model reappears for another model at a different budget. This groups all (budget, model, ratio) triples whose ratios agree within a relative tolerance.

In [102]:
# each printed group = one ratio value reached by several DIFFERENT (budget, model) pairs.
# read a line as: "budget B on model X gives ~ratio, and separately, budget B' on model Y ALSO gives ~ratio"
for group in find_ratio_coincidences(BUDGETS_B, MODEL_SIZES_B, rel_tol=0.01):
    print(f"ratio ~{group[0][0]:.3f}x is reached by:")
    for r, budget_b, name in group:
        print(f"    {budget_b:.3f}B tokens on {name}  (exact: {r:.4f}x)")

ratio ~0.179x is reached by:
    28.677B tokens on 8.031B  (exact: 0.1785x)
    12.763B tokens on 3.563B (TBD)  (exact: 0.1791x)
ratio ~0.400x is reached by:
    64.250B tokens on 8.031B  (exact: 0.4000x)
    28.677B tokens on 3.563B (TBD)  (exact: 0.4024x)
ratio ~0.515x is reached by:
    82.763B tokens on 8.031B  (exact: 0.5153x)
    12.763B tokens on 1.236B  (exact: 0.5163x)
    83.000B tokens on 8.031B  (exact: 0.5167x)
ratio ~0.900x is reached by:
    144.564B tokens on 8.031B  (exact: 0.9000x)
    64.250B tokens on 3.563B (TBD)  (exact: 0.9015x)
ratio ~1.160x is reached by:
    28.677B tokens on 1.236B  (exact: 1.1600x)
    82.763B tokens on 3.563B (TBD)  (exact: 1.1613x)
    83.000B tokens on 3.563B (TBD)  (exact: 1.1646x)
ratio ~1.723x is reached by:
    122.763B tokens on 3.563B (TBD)  (exact: 1.7225x)
    42.763B tokens on 1.236B  (exact: 1.7298x)
ratio ~3.348x is reached by:
    82.763B tokens on 1.236B  (exact: 3.3478x)
    83.000B tokens on 1.236B  (exact: 3.3574x)


## Systematically finding more magic budgets

Starting from the 1.236B model's real anchor budget (42.7628B), generates every budget reachable by chaining up to `max_hops` multiplications by the 3 model-size ratios (or their reciprocals), then reports which ones actually coincide.

In [109]:
ANCHOR_BUDGETS = [42.768 *1e9, 160.626 * 1e9]  # sunk 1.236B model's budget, and 1.0x for the 8.031B model
MIN_BUDGET = GUTENBERG_BUDGET
MAX_BUDGET = 160e9 + GUTENBERG_BUDGET

candidate_budgets = generate_magic_budgets(
    ANCHOR_BUDGETS, MODEL_SIZES_B, MIN_BUDGET, MAX_BUDGET, max_hops=1,
)
print(f"{len(candidate_budgets)} candidates: {[round(b/1e9, 3) for b in candidate_budgets]}")

show_cells = show_cells_from_coincidences(candidate_budgets, MODEL_SIZES_B)

# also force-show every cell that's ~1.0x on its own, even if it doesn't coincide with anything else
for b in candidate_budgets:
    for name, n in MODEL_SIZES_B.items():
        if abs(ratio(b, n) - 1.0) < 0.01:
            cell = show_cells.setdefault(b / 1e9, [])
            if name not in cell:
                cell.append(name)

make_ratio_table(candidate_budgets, MODEL_SIZES_B, show_cells=show_cells)

10 candidates: [6.582, 14.835, 18.976, 24.722, 42.768, 55.717, 71.27, 96.389, 123.296, 160.626]


,1.236B,3.563B (TBD),8.031B
budget (B tokens),,,
6.5823,0.27x,0.09x,
14.8350,0.60x,,0.09x
18.9762,0.77x,0.27x,
24.7215,1.00x,0.35x,
42.7680,1.73x,0.60x,0.27x
55.7166,2.25x,,0.35x
71.2699,,1.00x,
96.3893,,,0.60x
123.2963,,1.73x,0.77x


In [129]:
nn = [0.27, 1, 1.73, 3.19]

[1, ]
for i in range(len(nn) - 1):
    print(nn[i + 1] - nn[i])
3.19+0.73+0.73+0.73

0.73
0.73
1.46


4.65

## Completing partial coincidences

If 2 of the 3 models already share a ratio at their own budgets, this computes the 3rd model's matching budget (mean of the group's exact ratios x 20 x N) and adds it as a new row, as long as it falls within the valid budget range.

In [112]:
completed_budgets, show_cells = complete_coincidence_groups(
    BUDGETS_B, MODEL_SIZES_B, GUTENBERG_BUDGET, 160e9 + GUTENBERG_BUDGET,
)
print(f"{len(BUDGETS_B)} -> {len(completed_budgets)} budgets")

make_ratio_table(completed_budgets, MODEL_SIZES_B, show_cells=show_cells)

11 -> 12 budgets


,1.236B,3.563B (TBD),8.031B
budget (B tokens),,,
2.7628,,,
4.7628,,,
7.7628,,,
12.7628,0.52x,,
22.7628,,,
32.7628,,,
36.7580,,0.52x,
42.7628,1.73x,,
62.7628,,,
